# CatAI base pretraining on a Kaggle GPU

Runs the existing full 850-million-token base-training budget on CUDA. It does not reduce the corpus, token budget, context length, model preset, epoch count, or optimizer schedule. A complete run may exceed a Kaggle session; keep the recovery checkpoint and resume in a later session if needed.

In [ ]:
from pathlib import Path
import base64, os, subprocess, sys, time
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. In notebook Settings → Session options, set Accelerator to GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)


In [ ]:
# Kaggle Secrets → Add GITHUB_TOKEN with read-only access to bananakitssu/catai.
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
repo_dir = Path("/kaggle/working/catai")
if not repo_dir.exists():
    git_env = os.environ.copy()
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"})
    result = subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", "https://github.com/bananakitssu/catai.git", str(repo_dir)], env=git_env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Could not clone CatAI. Check that GITHUB_TOKEN has read-only access to the private repository.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(repo_dir) + "[data]"], check=True)
os.chdir(repo_dir)


In [ ]:
# Reuse a corpus attached as the private Kaggle Dataset catai-pretraining-corpus,
# or download the same curated corpus as the GitHub Actions workflow.
attached_corpus = Path("/kaggle/input/catai-pretraining-corpus/corpus.txt")
corpus = attached_corpus if attached_corpus.is_file() else Path("/kaggle/working/corpus.txt")
if not corpus.is_file():
    subprocess.run([
        sys.executable, "data/download_corpus.py",
        "--output", str(corpus),
        "--target-chars", "1400000000",
        "--fineweb-edu-chars", "210000000",
        "--cosmopedia-chars", "140000000",
        "--common-corpus-chars", "140000000",
        "--common-corpus-max-documents", "7000",
    ], cwd=repo_dir, check=True)
print("Corpus:", corpus, f"({corpus.stat().st_size / (1024**3):.2f} GiB)")


In [ ]:
# Exact base-training settings from the CI workflow; only CPU → CUDA changes.
command = [
    sys.executable, "-m", "catai",
    "--corpus", str(corpus),
    "--checkpoint", "/kaggle/working/catai.pt",
    "--recovery-checkpoint", "/kaggle/working/catai-base-recovery.pt",
    "--checkpoint-every-batches", "50",
    "--epochs", "1",
    "--batch-size", "4",
    "--sequence-length", "1024",
    "--model-size", "base",
    "--tokenizer", "bpe",
    "--vocab-size", "2048",
    "--tokenizer-sample-chars", "250000",
    "--max-tokens", "850000000",
    "--validation-split", "0.05",
    "--minimum-tokens-per-parameter", "20",
    "--warmup-fraction", "0.03",
    "--minimum-learning-rate-ratio", "0.1",
    "--seed", "42",
    "--device", "cuda",
    "--precision", "auto",
]
resume = Path("/kaggle/input/catai-pretraining-corpus/catai-base-recovery.pt")
if resume.is_file():
    command.extend(["--resume-from", str(resume)])
started = time.perf_counter()
subprocess.run(command, cwd=repo_dir, check=True)
torch.cuda.synchronize()
print(f"Training elapsed: {(time.perf_counter() - started) / 3600:.2f} hours")
print("Outputs: /kaggle/working/catai.pt and /kaggle/working/catai-base-recovery.pt")
